# 테스트·린트·빌드와 GitHub Actions

같은 검증 명령을 로컬과 CI에서 실행하고 종료 코드를 확인합니다.

### 이 준비 셀이 하는 일

첫 셀의 `Path` 탐색은 어느 상위 폴더에서 열어도 Ch33의 `pyproject.toml`을 기준으로 실습 폴더를 찾습니다. 뒤의 `subprocess.run(..., cwd=lab, capture_output=True)`은 명령을 별도 프로세스로 실행하고 화면 문구와 종료 코드를 함께 반환합니다. 이 보조 코드는 CI를 흉내 내는 공통 실행 경계이며, 각 절의 판정은 다음 셀에서 해당 명령의 실제 반환값을 읽어 결정합니다.

In [1]:
from pathlib import Path
import subprocess, sys
lab = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "ch33-cicd" and (p / "pyproject.toml").is_file()), Path("labs/part7/ch33-cicd")).resolve()
print(sys.version.split()[0], lab.exists())

3.12.13 True


### 바로 다음 코드를 읽는 방법

CI의 한 단계는 명령의 화면 문구보다 종료 코드로 성공과 실패를 판단합니다. pytest, Ruff, wheel 빌드는 각각 동작, 정적 규칙, 배포 산출물을 검사하며 서로 대신하지 않습니다. `subprocess.run`의 `returncode`가 0이면 성공이고 0이 아니면 실패입니다. 노트북은 일부러 실패하는 임시 테스트와 수정한 테스트를 같은 방식으로 실행해 이 계약을 확인합니다.

In [2]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=lab, text=True, capture_output=True)
print(result.stdout.strip())
print("exit_code=", result.returncode)
assert result.returncode == 0

.....                                                                    [100%]
5 passed in 0.07s
exit_code= 0


In [3]:
result = subprocess.run(["ruff", "check", "cicd_demo.py", "tests"], cwd=lab, text=True, capture_output=True)
print(result.stdout.strip() or "ruff: PASS")
print("exit_code=", result.returncode)
assert result.returncode == 0

All checks passed!
exit_code= 0


In [4]:
import tempfile
with tempfile.TemporaryDirectory() as out:
    result = subprocess.run([sys.executable, "-m", "build", "--wheel", "--outdir", out], cwd=lab, text=True, capture_output=True)
    wheels = [p.name for p in Path(out).glob("*.whl")]
    print("wheels=", wheels, "exit_code=", result.returncode)
    assert result.returncode == 0 and len(wheels) == 1

wheels= ['ch33_cicd_lab-0.1.0-py3-none-any.whl'] exit_code= 0


## 실패 재현

임시 저장소에서 실패하는 테스트를 추가합니다. 실제 pytest 프로세스가 0이 아닌 종료 코드를 반환합니다.

In [5]:
import tempfile
with tempfile.TemporaryDirectory() as temp:
    p = Path(temp) / "test_regression.py"
    p.write_text("def test_regression():\n    assert 2 + 2 == 5\n")
    failed = subprocess.run([sys.executable, "-m", "pytest", "-q", str(p)], text=True, capture_output=True)
    print(failed.stdout.strip().splitlines()[-1])
    print("exit_code=", failed.returncode)
    assert failed.returncode != 0

1 failed in 0.01s
exit_code= 1


## 수정

같은 테스트의 기대값을 고치면 프로세스가 종료 코드 0을 반환합니다.

### 이 단계에서 구분할 범위

로컬에서 세 명령이 통과해도 CI 환경의 Python 버전, 잠금 파일, 운영체제가 다르면 결과가 달라질 수 있습니다. 워크플로에는 실행 환경과 설치 명령을 고정하고, 빌드한 wheel을 다음 배포 단계가 같은 산출물로 사용하도록 보관합니다. 테스트를 배포 때 다시 다른 소스에서 만들면 검증 대상과 배포 대상이 어긋납니다.

In [6]:
with tempfile.TemporaryDirectory() as temp:
    p = Path(temp) / "test_regression.py"
    p.write_text("def test_regression():\n    assert 2 + 2 == 4\n")
    passed = subprocess.run([sys.executable, "-m", "pytest", "-q", str(p)], text=True, capture_output=True)
    print(passed.stdout.strip().splitlines()[-1])
    print("exit_code=", passed.returncode)
    assert passed.returncode == 0

1 passed in 0.00s
exit_code= 0
